In [ ]:
!pip -q install plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Upload CSV
df = pd.read_csv("Spotify_Analytics.csv")

print(f"Dataset shape: {df.shape}")
display(df.head())

Dataset shape: (500, 14)


,Artist Name,Sex,Country of Origin,Primary Language,Primary Genre,Artist Type,Debut Year,Total Streams (in millions),Lead Streams (in millions),Feature Streams (in millions),Solo Streams (in millions),% of Solo Streams,Collaborative Streams (in millions),% of Collaborative Streams
0,Drake,Male,Canada,English,Hip-Hop,Solo,2006,137492.1,94851.1,42640.9,53323.9,38.783246,84168.2,61.216754
1,Taylor Swift,Female,United States,English,Pop,Solo,2006,127861.0,126177.3,1683.8,113421.5,88.706877,14439.5,11.293123
2,Bad Bunny,Male,United States,Spanish,Reggaeton,Solo,2013,125899.8,82189.3,43710.5,48887.0,38.830086,77012.8,61.169914
3,The Weeknd,Male,Canada,English,R&B,Solo,2009,95703.2,77189.4,18513.8,51020.0,53.310652,44683.2,46.689348
4,Justin Bieber,Male,Canada,English,Pop,Solo,2009,78104.4,48316.2,29788.2,29638.3,37.947030,48466.1,62.052970


In [ ]:
# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace('\xa0', ' ', regex=False)
)

# Rename columns
df = df.rename(columns={
    'Artist Name': 'Artist',
    'Country of Origin': 'Country',
    'Primary Language': 'Language',
    'Primary Genre': 'Genre',
    'Artist Type': 'Artist Type',
    'Total Streams (in millions)': 'Total Streams',
    'Lead Streams (in millions)': 'Lead Streams',
    'Feature Streams (in millions)': 'Feature Streams',
    'Solo Streams (in millions)': 'Solo Streams',
    '% of Solo Streams': 'Solo %',
    'Collaborative Streams (in millions)': 'Collaborative Streams',
    '% of Collaborative Streams': 'Collaborative %'
})

numeric_cols = [
    'Debut Year',
    'Total Streams',
    'Lead Streams',
    'Feature Streams',
    'Solo Streams',
    'Solo %',
    'Collaborative Streams',
    'Collaborative %'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(how='all').reset_index(drop=True)

print("Dataset ready!")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

display(df.head())

Dataset ready!
Rows: 500
Columns: 14


,Artist,Sex,Country,Language,Genre,Artist Type,Debut Year,Total Streams,Lead Streams,Feature Streams,Solo Streams,Solo %,Collaborative Streams,Collaborative %
0,Drake,Male,Canada,English,Hip-Hop,Solo,2006,137492.1,94851.1,42640.9,53323.9,38.783246,84168.2,61.216754
1,Taylor Swift,Female,United States,English,Pop,Solo,2006,127861.0,126177.3,1683.8,113421.5,88.706877,14439.5,11.293123
2,Bad Bunny,Male,United States,Spanish,Reggaeton,Solo,2013,125899.8,82189.3,43710.5,48887.0,38.830086,77012.8,61.169914
3,The Weeknd,Male,Canada,English,R&B,Solo,2009,95703.2,77189.4,18513.8,51020.0,53.310652,44683.2,46.689348
4,Justin Bieber,Male,Canada,English,Pop,Solo,2009,78104.4,48316.2,29788.2,29638.3,37.947030,48466.1,62.052970


In [ ]:
# ============================================================
# VISUAL THEME
# ============================================================

COLORS = {
    "background": "#080B14",
    "card": "#111827",
    "card2": "#151D2E",
    "text": "#F8FAFC",
    "muted": "#94A3B8",
    "green": "#1DB954",
    "purple": "#8B5CF6",
    "pink": "#EC4899",
    "blue": "#38BDF8",
    "orange": "#F59E0B",
    "red": "#EF4444"
}

GENRE_COLORS = {
    "Pop": "#EC4899",
    "Hip-Hop": "#8B5CF6",
    "R&B": "#38BDF8",
    "Reggaeton": "#F59E0B",
    "Rock": "#EF4444",
    "K-Pop": "#1DB954",
    "Country": "#84CC16",
    "Electronic": "#06B6D4",
    "Latin": "#F97316"
}

display(HTML(f"""
<style>

body {{
    background-color: {COLORS["background"]};
}}

.dashboard {{
    background: {COLORS["background"]};
    color: {COLORS["text"]};
    padding: 20px;
    font-family: Arial, sans-serif;
}}

.dashboard-title {{
    font-size: 34px;
    font-weight: 800;
    color: white;
    margin-bottom: 5px;
}}

.dashboard-subtitle {{
    color: {COLORS["muted"]};
    font-size: 15px;
    margin-bottom: 25px;
}}

.kpi-container {{
    display: flex;
    gap: 15px;
    margin: 20px 0;
}}

.kpi {{
    flex: 1;
    padding: 22px;
    border-radius: 18px;
    background: linear-gradient(
        135deg,
        #151D2E,
        #111827
    );
    border: 1px solid #263044;
}}

.kpi-label {{
    color: {COLORS["muted"]};
    font-size: 12px;
    font-weight: 600;
    letter-spacing: 1px;
}}

.kpi-value {{
    color: white;
    font-size: 27px;
    font-weight: 800;
    margin-top: 7px;
}}

.section {{
    background: {COLORS["card"]};
    border-radius: 18px;
    padding: 15px;
    margin-top: 20px;
    border: 1px solid #202A3D;
}}

.section-title {{
    color: white;
    font-size: 18px;
    font-weight: 700;
}}

</style>

<div class="dashboard">

<div class="dashboard-title">
🎵 STREAMVERSE
</div>

<div class="dashboard-subtitle">
Global Artist Streaming Intelligence
</div>

</div>
"""))

In [ ]:
# ============================================================
# CELL 5 — VISUAL DASHBOARD
# No ipywidgets required
# ============================================================

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# ------------------------------------------------------------
# COLORS
# ------------------------------------------------------------

BG = "#080B14"
CARD = "#111827"
TEXT = "#F8FAFC"
MUTED = "#94A3B8"

GENRE_COLORS = {
    "Pop": "#EC4899",
    "Hip-Hop": "#8B5CF6",
    "R&B": "#38BDF8",
    "Reggaeton": "#F59E0B",
    "Rock": "#EF4444",
    "K-Pop": "#1DB954",
    "Country": "#84CC16",
    "Electronic": "#06B6D4",
    "Latin": "#F97316"
}

# ------------------------------------------------------------
# BASIC CHECK
# ------------------------------------------------------------

required_columns = [
    "Artist",
    "Genre",
    "Country",
    "Language",
    "Debut Year",
    "Total Streams",
    "Solo Streams",
    "Collaborative Streams"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:

    print("❌ Missing columns:")
    print(missing)

else:

    print("✅ Dashboard data loaded")
    print(f"Artists: {len(df)}")

    # ========================================================
    # HEADER
    # ========================================================

    display(HTML(f"""
    <div style="
        background:linear-gradient(
            135deg,
            #111827 0%,
            #312E81 50%,
            #831843 100%
        );
        padding:35px;
        border-radius:22px;
        color:white;
        margin:15px 0 25px 0;
    ">

        <div style="
            font-size:14px;
            letter-spacing:3px;
            color:#C4B5FD;
            font-weight:bold;
        ">
            GLOBAL MUSIC ANALYTICS
        </div>

        <div style="
            font-size:38px;
            font-weight:900;
            margin-top:8px;
        ">
            🎵 STREAMVERSE
        </div>

        <div style="
            color:#CBD5E1;
            font-size:16px;
            margin-top:8px;
        ">
            Exploring artists, genres, streaming behaviour
            and collaboration patterns
        </div>

    </div>
    """))


    # ========================================================
    # KPI DATA
    # ========================================================

    artist_count = len(df)

    total_streams = df["Total Streams"].sum()

    average_streams = df["Total Streams"].mean()

    median_streams = df["Total Streams"].median()

    top_artist = (
        df.loc[
            df["Total Streams"].idxmax(),
            "Artist"
        ]
    )

    top_streams = df["Total Streams"].max()


    # ========================================================
    # KPI CARDS
    # ========================================================

    display(HTML(f"""

    <div style="
        display:grid;
        grid-template-columns:
        repeat(4,1fr);
        gap:15px;
        margin-bottom:30px;
    ">

        <div style="
            background:#111827;
            border:1px solid #273449;
            border-radius:18px;
            padding:22px;
            color:white;
        ">
            <div style="color:#94A3B8;font-size:12px;">
                ARTISTS
            </div>

            <div style="
                font-size:30px;
                font-weight:800;
                margin-top:8px;
            ">
                {artist_count:,}
            </div>
        </div>


        <div style="
            background:#111827;
            border:1px solid #273449;
            border-radius:18px;
            padding:22px;
            color:white;
        ">
            <div style="color:#94A3B8;font-size:12px;">
                TOTAL STREAMS
            </div>

            <div style="
                font-size:30px;
                font-weight:800;
                margin-top:8px;
            ">
                {total_streams:,.0f}M
            </div>
        </div>


        <div style="
            background:#111827;
            border:1px solid #273449;
            border-radius:18px;
            padding:22px;
            color:white;
        ">
            <div style="color:#94A3B8;font-size:12px;">
                AVG STREAMS
            </div>

            <div style="
                font-size:30px;
                font-weight:800;
                margin-top:8px;
            ">
                {average_streams:,.0f}M
            </div>
        </div>


        <div style="
            background:
                linear-gradient(
                    135deg,
                    #7C3AED,
                    #EC4899
                );
            border-radius:18px;
            padding:22px;
            color:white;
        ">
            <div style="color:#E9D5FF;font-size:12px;">
                #1 ARTIST
            </div>

            <div style="
                font-size:25px;
                font-weight:800;
                margin-top:8px;
            ">
                {top_artist}
            </div>

            <div style="
                color:#F5D0FE;
                margin-top:5px;
            ">
                {top_streams:,.0f}M streams
            </div>
        </div>

    </div>

    """))


    # ========================================================
    # TOP 15 ARTISTS
    # ========================================================

    top_artists = (
        df
        .sort_values(
            "Total Streams",
            ascending=False
        )
        .head(15)
        .sort_values("Total Streams")
    )

    fig1 = px.bar(
        top_artists,
        x="Total Streams",
        y="Artist",
        orientation="h",
        color="Genre",
        color_discrete_map=GENRE_COLORS,
        hover_name="Artist",
        hover_data=[
            "Genre",
            "Country",
            "Debut Year",
            "Solo %",
            "Collaborative %"
        ]
    )

    fig1.update_layout(
        title="🏆 Top 15 Artists",
        paper_bgcolor=BG,
        plot_bgcolor=CARD,
        font=dict(color=TEXT),
        height=600,
        margin=dict(l=30,r=30,t=70,b=30),
        xaxis_title="Total Streams (millions)",
        yaxis_title=""
    )

    fig1.show()


    # ========================================================
    # GENRE + COUNTRY
    # ========================================================

    genre_data = (
        df
        .groupby("Genre", as_index=False)
        ["Total Streams"]
        .sum()
        .sort_values(
            "Total Streams",
            ascending=False
        )
    )

    fig2 = px.pie(
        genre_data,
        names="Genre",
        values="Total Streams",
        hole=.55,
        color="Genre",
        color_discrete_map=GENRE_COLORS
    )

    fig2.update_layout(
        title="🎼 Streaming by Genre",
        paper_bgcolor=BG,
        plot_bgcolor=CARD,
        font=dict(color=TEXT),
        height=500
    )

    fig2.show()


    # ========================================================
    # COUNTRY
    # ========================================================

    country_data = (
        df
        .groupby("Country", as_index=False)
        ["Total Streams"]
        .sum()
        .sort_values(
            "Total Streams",
            ascending=False
        )
        .head(15)
        .sort_values("Total Streams")
    )

    fig3 = px.bar(
        country_data,
        x="Total Streams",
        y="Country",
        orientation="h",
        color="Total Streams",
        color_continuous_scale=[
            "#06B6D4",
            "#6366F1",
            "#A855F7",
            "#EC4899"
        ]
    )

    fig3.update_layout(
        title="🌎 Top 15 Countries by Streams",
        paper_bgcolor=BG,
        plot_bgcolor=CARD,
        font=dict(color=TEXT),
        height=500,
        coloraxis_showscale=False
    )

    fig3.show()


    # ========================================================
    # ARTIST LANDSCAPE
    # ========================================================

    fig4 = px.scatter(
        df,
        x="Debut Year",
        y="Total Streams",
        size="Total Streams",
        color="Genre",
        color_discrete_map=GENRE_COLORS,
        hover_name="Artist",
        hover_data=[
            "Country",
            "Language",
            "Artist Type",
            "Solo %",
            "Collaborative %"
        ],
        size_max=45
    )

    fig4.update_layout(
        title="📈 Artist Landscape",
        paper_bgcolor=BG,
        plot_bgcolor=CARD,
        font=dict(color=TEXT),
        height=600,
        xaxis_title="Debut Year",
        yaxis_title="Total Streams (millions)"
    )

    fig4.show()


    # ========================================================
    # SOLO VS COLLABORATIVE
    # ========================================================

    collaboration = (
        df[
            [
                "Artist",
                "Solo Streams",
                "Collaborative Streams"
            ]
        ]
        .assign(
            Total=lambda x:
            x["Solo Streams"] +
            x["Collaborative Streams"]
        )
        .sort_values(
            "Total",
            ascending=False
        )
        .head(15)
        .sort_values("Total")
    )

    fig5 = go.Figure()

    fig5.add_trace(
        go.Bar(
            y=collaboration["Artist"],
            x=collaboration["Solo Streams"],
            name="Solo",
            orientation="h",
            marker_color="#8B5CF6"
        )
    )

    fig5.add_trace(
        go.Bar(
            y=collaboration["Artist"],
            x=collaboration["Collaborative Streams"],
            name="Collaborative",
            orientation="h",
            marker_color="#EC4899"
        )
    )

    fig5.update_layout(
        barmode="stack",
        title="🤝 Solo vs Collaborative Streams",
        paper_bgcolor=BG,
        plot_bgcolor=CARD,
        font=dict(color=TEXT),
        height=600,
        xaxis_title="Streams (millions)",
        yaxis_title=""
    )

    fig5.show()


    # ========================================================
    # STREAM COMPOSITION
    # ========================================================

    composition = pd.DataFrame({
        "Type": [
            "Lead Streams",
            "Feature Streams"
        ],
        "Streams": [
            df["Lead Streams"].sum(),
            df["Feature Streams"].sum()
        ]
    })

    fig6 = px.pie(
        composition,
        names="Type",
        values="Streams",
        hole=.55,
        color="Type",
        color_discrete_sequence=[
            "#8B5CF6",
            "#38BDF8"
        ]
    )

    fig6.update_layout(
        title="🎤 Lead vs Feature Streams",
        paper_bgcolor=BG,
        plot_bgcolor=CARD,
        font=dict(color=TEXT),
        height=450
    )

    fig6.show()


    # ========================================================
    # DATA SUMMARY
    # ========================================================

    display(HTML("""
    <div style="
        background:#111827;
        border-radius:18px;
        padding:25px;
        margin-top:20px;
        color:white;
    ">

        <h2 style="margin-top:0;">
            📊 Dataset Summary
        </h2>

        <p style="color:#94A3B8;">
            Interactive charts above can be hovered,
            zoomed, filtered through legends, and downloaded.
        </p>

    </div>
    """))

✅ Dashboard data loaded
Artists: 500


In [ ]:
# ============================================================
# ARTIST EXPLORER
# ============================================================

artist_dropdown = widgets.Dropdown(
    options=sorted(df["Artist"].dropna().unique()),
    description="Artist:",
    layout=widgets.Layout(width="400px")
)

artist_output = widgets.Output()


def show_artist(change=None):

    with artist_output:

        clear_output(wait=True)

        artist = artist_dropdown.value

        row = df[df["Artist"] == artist].iloc[0]

        total = row["Total Streams"]
        solo = row["Solo Streams"]
        collab = row["Collaborative Streams"]

        display(HTML(f"""

        <div style="
            background:linear-gradient(
                135deg,
                #151D2E,
                #312E81
            );
            padding:30px;
            border-radius:22px;
            color:white;
            margin-top:20px;
        ">

            <div style="
                font-size:14px;
                color:#A5B4FC;
                font-weight:bold;
                letter-spacing:2px;
            ">
            ARTIST SPOTLIGHT
            </div>

            <div style="
                font-size:32px;
                font-weight:800;
                margin-top:5px;
            ">
            {artist}
            </div>

            <div style="
                color:#CBD5E1;
                margin-top:5px;
            ">
            {row["Genre"]} · {row["Country"]} ·
            {row["Language"]}
            </div>

            <hr style="
                border-color:#4C1D95;
                margin:25px 0;
            ">

            <div style="
                display:flex;
                gap:50px;
            ">

                <div>
                    <div style="color:#A5B4FC;">
                    TOTAL STREAMS
                    </div>

                    <div style="
                        font-size:28px;
                        font-weight:bold;
                    ">
                    {total:,.0f}M
                    </div>
                </div>

                <div>
                    <div style="color:#A5B4FC;">
                    DEBUT
                    </div>

                    <div style="
                        font-size:28px;
                        font-weight:bold;
                    ">
                    {int(row["Debut Year"])}
                    </div>
                </div>

                <div>
                    <div style="color:#A5B4FC;">
                    SOLO %
                    </div>

                    <div style="
                        font-size:28px;
                        font-weight:bold;
                    ">
                    {row["Solo %"]:.1f}%
                    </div>
                </div>

            </div>

        </div>

        """))


        # Stream composition

        fig = go.Figure()

        fig.add_trace(
            go.Bar(
                x=[solo],
                y=["Streams"],
                orientation="h",
                name="Solo",
                marker_color=COLORS["purple"]
            )
        )

        fig.add_trace(
            go.Bar(
                x=[collab],
                y=["Streams"],
                orientation="h",
                name="Collaborative",
                marker_color=COLORS["pink"]
            )
        )

        fig.update_layout(
            barmode="stack",
            title="Stream Composition",
            xaxis_title="Streams (millions)"
        )

        style_chart(fig, 300)

        fig.show()


artist_dropdown.observe(
    show_artist,
    names="value"
)

display(HTML("""
<h2 style="
    color:white;
    margin-top:35px;
">
🎤 Artist Explorer
</h2>
"""))

display(artist_dropdown)
display(artist_output)

show_artist()

Dropdown(description='Artist:', layout=Layout(width='400px'), options=('$uicideboy$', '2 Chainz', '21 Savage',…

Output()